# Vision RAG — ColQwen2.5 Embedding (Kaggle)

Safe to Run All. Skips steps that are already done.

| Index | Dataset | Pages | ~Time (P100) |
|-------|---------|-------|--------------|
| 0 | hr (start here) | 1,110 | ~17min |
| 1 | computer_science | 1,360 | ~21min |
| 2 | finance_en | 2,942 | ~45min |

**Enable GPU:** Settings (right sidebar) -> Accelerator -> GPU P100

In [ ]:
DATASET_INDEX = 0  # 0=hr, 1=CS, 2=finance

In [ ]:
%%time
!pip install -q --upgrade torchao>=0.16.0
!pip install -q colpali-engine>=0.3.0 datasets>=3.0.0 pytrec-eval-terrier>=0.5.0

In [ ]:
import os, json, time, gc
import torch
import numpy as np
from datasets import load_dataset
from PIL import Image

DATASETS = [
    ('hr', 'vidore/vidore_v3_hr'),
    ('computer_science', 'vidore/vidore_v3_computer_science'),
    ('finance_en', 'vidore/vidore_v3_finance_en'),
]

DATASET_KEY, DATASET_NAME = DATASETS[DATASET_INDEX]
OUTPUT_DIR = f'/kaggle/working/{DATASET_KEY}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_embeddings(embeddings_list, metadata, path):
    arrays = {f'page_{i}': emb for i, emb in enumerate(embeddings_list)}
    np.savez_compressed(path, **arrays)
    with open(path.replace('.npz', '_meta.json'), 'w') as f:
        json.dump(metadata, f)
    print(f'  Saved -> {path} ({os.path.getsize(path)/1e6:.1f} MB)')

def load_embeddings(path):
    data = np.load(path, allow_pickle=True)
    embeddings = [data[f'page_{i}'] for i in range(len(data.files))]
    with open(path.replace('.npz', '_meta.json')) as f:
        metadata = json.load(f)
    return embeddings, metadata

print(f'Dataset: {DATASET_KEY}')
print(f'Output: {OUTPUT_DIR}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Check what's already done
pages_done = os.path.exists(f'{OUTPUT_DIR}/pages.npz')
queries_done = os.path.exists(f'{OUTPUT_DIR}/queries.npz')
pruned_done = os.path.exists(f'{OUTPUT_DIR}/pages_pruned.npz')
qrels_done = os.path.exists(f'{OUTPUT_DIR}/qrels.json')
results_done = os.path.exists(f'{OUTPUT_DIR}/results.json')

print(f'Pages embedded: {pages_done}')
print(f'Queries embedded: {queries_done}')
print(f'Pruned: {pruned_done}')
print(f'Qrels saved: {qrels_done}')
print(f'Results: {results_done}')

NEED_MODEL = not pages_done or not queries_done
print(f'\nNeed to load model: {NEED_MODEL}')

In [ ]:
# Only load the heavy model if we need to embed something
if NEED_MODEL:
    from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor
    MODEL_NAME = 'vidore/colqwen2.5-v0.2'
    model = ColQwen2_5.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map='cuda'
    ).eval()
    processor = ColQwen2_5_Processor.from_pretrained(MODEL_NAME)
    print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
else:
    MODEL_NAME = 'vidore/colqwen2.5-v0.2'
    print('Model not needed - all embeddings already exist')

In [ ]:
# Load dataset metadata (always needed for IDs)
corpus_ds = load_dataset(DATASET_NAME, 'corpus', trust_remote_code=True)['test']
queries_ds = load_dataset(DATASET_NAME, 'queries', trust_remote_code=True)['test']

corpus_ids = [str(row['corpus_id']) for row in corpus_ds]
query_ids = [str(row['query_id']) for row in queries_ds]
query_texts = [row['query'] for row in queries_ds]

print(f'{len(corpus_ids)} pages, {len(query_texts)} queries')

In [ ]:
# === EMBED PAGES (skips if already done) ===
if pages_done:
    print(f'Pages already embedded, loading from {OUTPUT_DIR}/pages.npz')
    page_embeddings, page_meta = load_embeddings(f'{OUTPUT_DIR}/pages.npz')
    embed_time = 0
    print(f'Loaded {len(page_embeddings)} page embeddings, shape: {page_embeddings[0].shape}')
else:
    # Load images only when we need to embed
    corpus_images = [row['image'] for row in corpus_ds]
    page_embeddings = []
    t_start = time.time()
    BATCH = 2
    for i in range(0, len(corpus_images), BATCH):
        batch_imgs = corpus_images[i:i+BATCH]
        batch = processor.process_images(batch_imgs)
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.no_grad():
            embs = model(**batch)
        for emb in embs:
            page_embeddings.append(emb.cpu().float().numpy())
        del batch, embs
        torch.cuda.empty_cache()
        done = min(i+BATCH, len(corpus_images))
        if done % 100 < BATCH or done == len(corpus_images):
            elapsed = time.time() - t_start
            eta = (elapsed/done) * (len(corpus_images)-done) / 60
            print(f'{done}/{len(corpus_images)}  ({elapsed/60:.1f}min, ~{eta:.0f}min left)')
    embed_time = time.time() - t_start
    print(f'\nDone: {embed_time/60:.1f}min ({embed_time/len(corpus_images):.1f}s/page)')
    print(f'Shape: {page_embeddings[0].shape}')
    page_meta = [{'corpus_id': cid, 'index': i} for i, cid in enumerate(corpus_ids)]
    save_embeddings(page_embeddings, page_meta, f'{OUTPUT_DIR}/pages.npz')
    # Free images from memory
    del corpus_images
    gc.collect()

In [ ]:
# === EMBED QUERIES (skips if already done) ===
if queries_done:
    print(f'Queries already embedded, loading from {OUTPUT_DIR}/queries.npz')
    query_embeddings, query_meta = load_embeddings(f'{OUTPUT_DIR}/queries.npz')
    print(f'Loaded {len(query_embeddings)} query embeddings')
else:
    query_embeddings = []
    for i in range(0, len(query_texts), 8):
        batch = processor.process_queries(query_texts[i:i+8])
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.no_grad():
            embs = model(**batch)
        for emb in embs:
            query_embeddings.append(emb.cpu().float().numpy())
        del batch, embs
        torch.cuda.empty_cache()
    print(f'{len(query_embeddings)} queries embedded')
    query_meta = [{'query_id': qid, 'text': qt, 'index': i}
                  for i, (qid, qt) in enumerate(zip(query_ids, query_texts))]
    save_embeddings(query_embeddings, query_meta, f'{OUTPUT_DIR}/queries.npz')

# Free model from GPU memory - no longer needed
if NEED_MODEL:
    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print('Model freed from memory')

In [ ]:
# === PRUNE + QRELS (skips if already done) ===
if not pruned_done:
    pruned = []
    for emb in page_embeddings:
        norms = np.linalg.norm(emb, axis=1)
        k = max(1, int(len(norms) * 0.5))
        top_idx = np.argpartition(norms, -k)[-k:]
        top_idx.sort()
        pruned.append(emb[top_idx])
    raw_p = sum(e.shape[0] for e in page_embeddings)
    pru_p = sum(e.shape[0] for e in pruned)
    print(f'Pruned: {raw_p} -> {pru_p} patches ({pru_p/raw_p:.1%})')
    if not page_meta:
        page_meta = [{'corpus_id': cid, 'index': i} for i, cid in enumerate(corpus_ids)]
    save_embeddings(pruned, page_meta, f'{OUTPUT_DIR}/pages_pruned.npz')
else:
    raw_p = sum(e.shape[0] for e in page_embeddings)
    pruned, _ = load_embeddings(f'{OUTPUT_DIR}/pages_pruned.npz')
    pru_p = sum(e.shape[0] for e in pruned)
    print(f'Pruned embeddings loaded: {pru_p} patches')

# Free raw page embeddings - only pruned needed from here
del page_embeddings
gc.collect()

if not qrels_done:
    qrels_raw = load_dataset(DATASET_NAME, 'qrels', trust_remote_code=True)['test']
    qrels_list = []
    for row in qrels_raw:
        entry = {}
        for k, v in dict(row).items():
            entry[k] = v if isinstance(v, (str, int, float, bool, list, dict, type(None))) else str(v)
        qrels_list.append(entry)
    with open(f'{OUTPUT_DIR}/qrels.json', 'w') as f:
        json.dump(qrels_list, f)
    print(f'{len(qrels_list)} qrels saved')
else:
    print('Qrels already saved')

In [ ]:
# === EVALUATE ===
import pytrec_eval

page_embs, p_meta = load_embeddings(f'{OUTPUT_DIR}/pages_pruned.npz')
query_embs, q_meta = load_embeddings(f'{OUTPUT_DIR}/queries.npz')

with open(f'{OUTPUT_DIR}/qrels.json') as f:
    qrels_list = json.load(f)

qrels_dict = {}
for row in qrels_list:
    qid = str(row['query_id'])
    cid = str(row['corpus_id'])
    qrels_dict.setdefault(qid, {})[cid] = int(row['score'])

print(f'Scoring {len(query_embs)} queries x {len(page_embs)} pages...')
run_dict = {}
latencies = []

for qi, (q_emb, qm) in enumerate(zip(query_embs, q_meta)):
    qid = qm['query_id']
    if qid not in qrels_dict:
        continue
    t0 = time.time()
    scores = np.zeros(len(page_embs))
    for pi, pe in enumerate(page_embs):
        scores[pi] = (q_emb @ pe.T).max(axis=1).sum()
    latencies.append(time.time() - t0)
    top_idx = np.argpartition(scores, -10)[-10:]
    run_dict[qid] = {p_meta[idx]['corpus_id']: float(scores[idx]) for idx in top_idx}
    if (qi+1) % 200 == 0:
        print(f'  {qi+1}/{len(query_embs)}')

evaluator = pytrec_eval.RelevanceEvaluator(
    qrels_dict, {'ndcg_cut_5', 'ndcg_cut_10', 'recall_5', 'recall_10', 'map'}
)
eval_results = evaluator.evaluate(run_dict)

metrics = {}
for m in ['ndcg_cut_5', 'ndcg_cut_10', 'recall_5', 'recall_10', 'map']:
    vals = [r[m] for r in eval_results.values()]
    metrics[m] = sum(vals) / len(vals) if vals else 0

lat = np.array(latencies) * 1000
metrics['retrieval_p50_ms'] = float(np.percentile(lat, 50))
metrics['retrieval_p95_ms'] = float(np.percentile(lat, 95))

print(f'\n{"=" * 50}')
print(f'  {DATASET_KEY} -- Vision RAG (ColQwen2.5, pruned 50%)')
print(f'{"=" * 50}')
print(f'  NDCG@5:     {metrics["ndcg_cut_5"]:.4f}')
print(f'  NDCG@10:    {metrics["ndcg_cut_10"]:.4f}')
print(f'  Recall@5:   {metrics["recall_5"]:.4f}')
print(f'  Recall@10:  {metrics["recall_10"]:.4f}')
print(f'  MAP:        {metrics["map"]:.4f}')
print(f'  p95 latency: {metrics["retrieval_p95_ms"]:.0f}ms')
print(f'{"=" * 50}')

In [ ]:
# === SAVE RESULTS ===
results = {
    'dataset': DATASET_KEY,
    'model': MODEL_NAME,
    'prune_fraction': 0.5,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'n_pages': len(corpus_ids),
    'n_queries': len(query_texts),
    'embed_time_sec': embed_time,
    'raw_patches': raw_p,
    'pruned_patches': pru_p,
    'compression_ratio': raw_p / pru_p,
    **metrics,
}

with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print(f'\nAll files in: {OUTPUT_DIR}')
print('Download from Output tab at top of page')